In [1]:
!pip install -q transformers accelerate torch fastapi uvicorn pyngrok nest-asyncio pydantic

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "kritika53245/mindhaven-cbt-qwen"

print("Loading tokenizer and model (this might take a few minutes)...")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
print("Model loaded successfully on GPU!")

Loading tokenizer and model (this might take a few minutes)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.38k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/15.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded successfully on GPU!


In [6]:
import nest_asyncio
import uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import List, Dict, Optional

# Enable nesting uvicorn loops in Google Colab
nest_asyncio.apply()

app = FastAPI(title="MindHaven CBT Inference API")

# Enable CORS so your frontend can fetch directly from this tunnel
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class Message(BaseModel):
    role: str
    content: str

class ChatCompletionRequest(BaseModel):
    messages: List[Message]
    temperature: Optional[float] = 0.7
    max_tokens: Optional[int] = 512

@app.post("/v1/chat/completions")
async def chat_completions(request: ChatCompletionRequest):
    try:
        # Format the input using the Qwen chat template
        formatted_messages = [{"role": m.role, "content": m.content} for m in request.messages]

        # Apply Qwen template
        prompt = tokenizer.apply_chat_template(
            formatted_messages,
            tokenize=False,
            add_generation_prompt=True
        )

        # Tokenize and generate
        inputs = tokenizer([prompt], return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=request.max_tokens,
                temperature=request.temperature,
                do_sample=True if request.temperature > 0 else False,
                pad_token_id=tokenizer.eos_token_id
            )

        # Decode output
        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, outputs)
        ]
        response_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

        # Return standard OpenAI response format
        return {
            "choices": [
                {
                    "message": {
                        "role": "assistant",
                        "content": response_text.strip()
                    },
                    "finish_reason": "stop",
                    "index": 0
                }
            ]
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

In [ ]:
from pyngrok import ngrok

# Configure your token (replace with your copied token)
NGROK_TOKEN = ""
ngrok.set_auth_token(NGROK_TOKEN)

# Open an HTTP tunnel on port 8000
public_url = ngrok.connect(8000)
print("\n" + "="*60)
print(f"👉 Ngrok Tunnel is Active!")
print(f"🔗 Public OpenAI endpoint: {public_url.public_url}/v1/chat/completions")
print("="*60 + "\n")

# Start server
# Start server using the notebook's active event loop
config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
await server.serve()


👉 Ngrok Tunnel is Active!
🔗 Public OpenAI endpoint: https://cubicle-wildcard-backpedal.ngrok-free.dev/v1/chat/completions



INFO:     Started server process [3632]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     223.226.243.213:0 - "GET /v1/chat/completions HTTP/1.1" 405 Method Not Allowed
INFO:     223.226.243.213:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     223.226.243.213:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     223.226.243.213:0 - "POST /v1/chat/completions HTTP/1.1" 422 Unprocessable Entity
INFO:     223.226.243.213:0 - "POST /v1/chat/completions HTTP/1.1" 422 Unprocessable Entity
INFO:     223.226.243.213:0 - "POST /v1/chat/completions HTTP/1.1" 200 OK
INFO:     223.226.243.213:0 - "POST /v1/chat/completions HTTP/1.1" 200 OK
INFO:     223.226.243.213:0 - "POST /v1/chat/completions HTTP/1.1" 200 OK
